# Notebook de test des modules 

### Imports et configuration

In [8]:
import os
import sys
import glob
from tqdm.notebook import tqdm
import sacrebleu

from data_loader import load_mtedx_data
from asr_pipeline import AudioTranscriber
from nmt_pipeline import SubtitleTranslator
from metrics import evaluate_asr, evaluate_nmt

sys.path.append(os.path.abspath('./src'))

try:
    BASE_DIR = os.path.abspath(__file__)
except NameError:
    BASE_DIR = os.path.dirname(".")

DATA_DIR = os.path.join(os.path.abspath(BASE_DIR), 'data')

# Chemins spécifiques datasets test FR
FR_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-fr', 'data', 'test')
FR_WAV_TEST_DIR = os.path.join(FR_TEST_DIR, 'wav')
FR_TXT_TEST_DIR = os.path.join(FR_TEST_DIR, 'txt')
FR_VTT_TEST_DIR = os.path.join(FR_TEST_DIR, 'vtt')

# Chemins spécifiques datasets test EN
EN_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-en', 'data', 'test')
EN_WAV_TEST_DIR = os.path.join(EN_TEST_DIR, 'wav')
EN_TXT_TEST_DIR = os.path.join(EN_TEST_DIR, 'txt')
EN_VTT_TEST_DIR = os.path.join(EN_TEST_DIR, 'vtt')

# Chemins spécifiques datasets test ES
ES_TEST_DIR = os.path.join(os.path.abspath(DATA_DIR), 'fr-es', 'data', 'test')
ES_WAV_TEST_DIR = os.path.join(ES_TEST_DIR, 'wav')
ES_TXT_TEST_DIR = os.path.join(ES_TEST_DIR, 'txt')
ES_VTT_TEST_DIR = os.path.join(ES_TEST_DIR, 'vtt')


## Chargement des données

In [5]:
print(f"Chargement des données depuis {DATA_DIR}...")

datasets = load_mtedx_data(DATA_DIR, pairs=['fr-en','fr-es'])

print("\n---- Résultats du chargement ----")
print("Paires de langues chargées :", datasets.keys())

if 'fr-en' in datasets:
    print("\nSplits disponibles pour fr-en :", datasets['fr-en'].keys())
    print(f"Nombre de segments audio (train): {len(datasets['fr-en']['train'])}")

    print("\nExemple de segment audio (segment 0) :")
    exemple = datasets['fr-en']['train'][0]
    for key, value in exemple.items():
        print(f"{key}: {value[:100]}...")

Chargement des données depuis /home/dylan/COURS/M2_COURS/deep_learning/projet_deeplearning/data...
fr-en - train chargé : 30171 phrases.
fr-en - valid chargé : 1036 phrases.
fr-en - test chargé : 1059 phrases.
fr-es - train chargé : 20826 phrases.
fr-es - valid chargé : 1036 phrases.
fr-es - test chargé : 1059 phrases.

---- Résultats du chargement ----
Paires de langues chargées : dict_keys(['fr-en', 'fr-es'])

Splits disponibles pour fr-en : dict_keys(['train', 'valid', 'test'])
Nombre de segments audio (train): 30171

Exemple de segment audio (segment 0) :
src_text: Je m'appelle Julien Le Breton et, comme mon nom ne l'indique pas, je suis né et j'habite en Nouvelle...
tgt_text: I'm Julien Le Breton and as my name doesn't suggest, I was born and live in New Caledonia....


## Transcription sur un fichier audio

### Transcription et Evaluation de l'ASR

In [6]:
print("=== ÉVALUATION ASR (Français) ===")

segments_file = os.path.join(FR_TXT_TEST_DIR, "segments")
text_fr_file = os.path.join(FR_TXT_TEST_DIR, "test.fr")

references_par_audio = {}
if os.path.exists(segments_file) and os.path.exists(text_fr_file):
    with open(segments_file, 'r', encoding='utf-8') as f_seg, open(text_fr_file, 'r', encoding='utf-8') as f_txt:
        segments_lines = f_seg.readlines()
        text_lines = f_txt.readlines()
        
        # On associe chaque ligne de test.fr à son identifiant audio grâce au fichier segments
        for seg_line, txt_line in zip(segments_lines, text_lines):
            parts = seg_line.strip().split()
            if len(parts) >= 2:
                recording_id = parts[1] # ex: '0u7tTptBo9I'
                texte = txt_line.strip()
                if recording_id not in references_par_audio:
                    references_par_audio[recording_id] = []
                references_par_audio[recording_id].append(texte)

# B. Joindre les phrases pour faire un texte complet par fichier audio
for rec_id in references_par_audio:
    references_par_audio[rec_id] = " ".join(references_par_audio[rec_id])

# C. Lancer Whisper et évaluer
transcripteur = AudioTranscriber(model_name="openai/whisper-small")
hypotheses_asr = []
references_asr = []

# LIMITE : On n'évalue que les 2 premiers fichiers audio pour le test. 
# Enlève `[:2]` pour évaluer tout le dataset (ça prendra du temps !)
fichiers_a_evaluer = list(references_par_audio.keys())[:2]

for rec_id in tqdm(fichiers_a_evaluer, desc="Transcription ASR"):
    # Tu as bien précisé que ce sont des .flac dans le dossier wav !
    audio_path = os.path.join(FR_WAV_TEST_DIR, f"{rec_id}.flac") 
    
    if os.path.exists(audio_path):
        # On utilise le pipeline pour extraire le texte complet
        result = transcripteur.pipe(audio_path, generate_kwargs={"language": "french", "task": "transcribe"})
        hypotheses_asr.append(result["text"])
        references_asr.append(references_par_audio[rec_id])
    else:
        print(f"Audio introuvable : {audio_path}")

# D. Calcul des scores ASR
if hypotheses_asr:
    scores_asr = evaluate_asr(references_asr, hypotheses_asr)
    print(f"Résultats ASR : WER = {scores_asr['WER']}%, CER = {scores_asr['CER']}%\n")
    

=== ÉVALUATION ASR (Français) ===
Chargement du modèle ASR 'openai/whisper-small' sur : cuda:0...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcription ASR:   0%|          | 0/2 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Whisper did not predict an endi

Résultats ASR : WER = 21.14%, CER = 17.42%



### Traduction et Evaluation du NMT

In [7]:
print("=== ÉVALUATION NMT (Français -> Anglais) ===")

nmt_src_file = os.path.join(EN_TXT_TEST_DIR, "test.fr")
nmt_tgt_file = os.path.join(EN_TXT_TEST_DIR, "test.en")

if os.path.exists(nmt_src_file) and os.path.exists(nmt_tgt_file):
    with open(nmt_src_file, 'r', encoding='utf-8') as f_src, open(nmt_tgt_file, 'r', encoding='utf-8') as f_tgt:
        phrases_fr = [line.strip() for line in f_src.readlines()]
        references_en = [line.strip() for line in f_tgt.readlines()]
        
    # ⚠️ LIMITE : On prend un sous-ensemble (50 phrases) pour que ça aille vite.
    EVAL_SIZE = 50
    phrases_fr = phrases_fr[:EVAL_SIZE]
    references_en = references_en[:EVAL_SIZE]
    
    traducteur_en = SubtitleTranslator(model_name="Helsinki-NLP/opus-mt-fr-en")
    
    hypotheses_en = []
    for phrase in tqdm(phrases_fr, desc="Traduction NMT"):
        hypotheses_en.append(traducteur_en.translate_text(phrase))
        
    scores_nmt = evaluate_nmt(references_en, hypotheses_en)
    print(f"Résultats NMT : BLEU = {scores_nmt['BLEU']}, chrF = {scores_nmt['chrF']}")
else:
    print(f"Fichiers de traduction introuvables dans {EN_TEST_DIR}")

=== ÉVALUATION NMT (Français -> Anglais) ===
Chargement du modèle NMT 'Helsinki-NLP/opus-mt-fr-en' sur : cuda:0...


/home/dylan/.cache/pypoetry/virtualenvs/projet-deeplearning-aNZGoVd5-py3.11/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Traduction NMT:   0%|          | 0/50 [00:00<?, ?it/s]

Résultats NMT : BLEU = 47.35, chrF = 67.89


## Erreurs en cascade (ASR puis NMT)

In [9]:
print("=== EXPÉRIENCE : LES ERREURS EN CASCADE (ASR -> NMT) ===")

# 1. Chargement des données pour le premier fichier audio
segments_file = os.path.join(FR_TXT_TEST_DIR, "segments")
test_fr_file = os.path.join(FR_TXT_TEST_DIR, "test.fr")
test_en_file = os.path.join(EN_TXT_TEST_DIR, "test.en")

# On récupère les identifiants pour grouper par audio
with open(segments_file, 'r', encoding='utf-8') as f:
    segments = f.readlines()
    
# On prend le tout premier ID de conférence (ex: '0u7tTptBo9I')
first_rec_id = segments[0].split()[1]
print(f"Analyse sur la vidéo ID : {first_rec_id}")

# On trouve quelles lignes du texte correspondent à cette vidéo
indices_lignes = [i for i, line in enumerate(segments) if line.split()[1] == first_rec_id]

with open(test_fr_file, 'r', encoding='utf-8') as f_fr, open(test_en_file, 'r', encoding='utf-8') as f_en:
    lignes_fr = f_fr.readlines()
    lignes_en = f_en.readlines()

# Texte parfait (Ground Truth)
phrases_gt_fr = [lignes_fr[i].strip() for i in indices_lignes]
phrases_gt_en = [lignes_en[i].strip() for i in indices_lignes]
texte_complet_ref_en = " ".join(phrases_gt_en) # La traduction humaine de référence

# 2. Étape ASR : Obtenir la transcription Whisper (avec ses fautes)
audio_path = os.path.join(FR_WAV_TEST_DIR, f"{first_rec_id}.flac")
transcripteur = AudioTranscriber(model_name="openai/whisper-small")

print("\n1/3 - Whisper écoute et transcrit l'audio (ASR)...")
resultat_asr = transcripteur.pipe(audio_path, generate_kwargs={"language": "french", "task": "transcribe"})
# On récupère les chunks (morceaux) pour ne pas saturer le traducteur d'un coup
chunks_asr_fr = [chunk['text'] for chunk in resultat_asr['chunks']]

# 3. Étape Traduction (NMT)
traducteur = SubtitleTranslator(model_name="Helsinki-NLP/opus-mt-fr-en")

print("2/3 - Traduction du texte PARFAIT (Ground Truth)...")
traductions_gt = [traducteur.translate_text(phrase) for phrase in phrases_gt_fr]
texte_complet_traduit_gt = " ".join(traductions_gt)

print("3/3 - Traduction de la sortie WHISPER (Cascade Error)...")
traductions_asr = [traducteur.translate_text(chunk) for chunk in chunks_asr_fr]
texte_complet_traduit_asr = " ".join(traductions_asr)

# 4. Évaluation (Comparaison des deux textes complets)
bleu_ideal = sacrebleu.sentence_bleu(texte_complet_traduit_gt, [texte_complet_ref_en]).score
bleu_cascade = sacrebleu.sentence_bleu(texte_complet_traduit_asr, [texte_complet_ref_en]).score

print("\n=== RÉSULTATS DE L'EXPÉRIENCE ===")
print(f"Score BLEU idéal (sans erreur ASR) : {round(bleu_ideal, 2)}")
print(f"Score BLEU en cascade (ASR + NMT)  : {round(bleu_cascade, 2)}")
print(f"Chute de qualité due à l'audio  : -{round(bleu_ideal - bleu_cascade, 2)} points BLEU")

print("\n=== ANALYSE QUALITATIVE (Extrait) ===")
print(f"Ce que la personne a VRAIMENT dit : {phrases_gt_fr[0]}")
print(f"Ce que Whisper a compris (ASR)   : {chunks_asr_fr[0].strip()}")
print(f"Traduction du texte parfait      : {traductions_gt[0]}")
print(f"Traduction de l'erreur (Cascade) : {traductions_asr[0]}")

=== EXPÉRIENCE : LES ERREURS EN CASCADE (ASR -> NMT) ===
Analyse sur la vidéo ID : 0u7tTptBo9I
Chargement du modèle ASR 'openai/whisper-small' sur : cuda:0...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



1/3 - Whisper écoute et transcrit l'audio (ASR)...


Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Chargement du modèle NMT 'Helsinki-NLP/opus-mt-fr-en' sur : cuda:0...


/home/dylan/.cache/pypoetry/virtualenvs/projet-deeplearning-aNZGoVd5-py3.11/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

2/3 - Traduction du texte PARFAIT (Ground Truth)...
3/3 - Traduction de la sortie WHISPER (Cascade Error)...

=== RÉSULTATS DE L'EXPÉRIENCE ===
Score BLEU idéal (sans erreur ASR) : 45.3
Score BLEU en cascade (ASR + NMT)  : 27.2
Chute de qualité due à l'audio  : -18.1 points BLEU

=== ANALYSE QUALITATIVE (Extrait) ===
Ce que la personne a VRAIMENT dit : Bonsoir !
Ce que Whisper a compris (ASR)   : ... Bonsoir. Notre planète est recouverte à 70 % d'océans, et pourtantrangement on a choisi de l'appeler la Terre. Le poète Edcott Williams a une vision bien plus objective et moins anthropocentrique quand il dit que vu de l'espace, la planète est bleue. Vu de l'espace, elle est le territoire non pas des hommes mais des baleines. Et pourtant, on vient tous de l'océan, c'est le berceau de la vie, même si on l'a oublié.
Traduction du texte parfait      : Good evening!
Traduction de l'erreur (Cascade) : ... Good evening. Our planet is covered with 70% oceans, and yet we have chosen to call it Ear

In [ ]:
## 